## Imports

In [1]:
import os
import pandas as pd, math
import numpy as np
from pyhive import presto
from datetime import datetime, date, timedelta
import warnings

# Ignore a specific warning by its type
warnings.filterwarnings("ignore")

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

## Connection

In [2]:
presto_port = '80'
# username = 'airflow-user'
username = 'manoj.ravirajan@rapido.bike'

connection1 = presto.connect('bi-presto.serving.data.production.internal', presto_port, username)
connection2 = presto.connect('bi-trino-2.serving.data.production.internal', presto_port, username)
connection3 = presto.connect('processing-2.processing.data.production.internal', presto_port, username)
connection4 = presto.connect('processing.processing.data.production.internal', presto_port, username)

## Parameter

In [3]:
city = 'Bangalore'
date_list = ['2024-04-07', '2024-04-14', '2024-04-21', '2024-04-28', '2024-05-05']
base_week = date_list[0]

## Dataset & User-Defined Functions

In [4]:
def consideration_query(run_date, city_name, base_week):
    
    rr_data_query = f"""
  
        with dormant_customer as (
            
            select
                customer_id as customer,
                date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) rapido_first_ride,
                ceil(cast(date_diff('day',taxi_lifetime_first_ride_date,date(run_date)) as real)/10)*10 rapido_age_seg_fr,
                
                taxi_lifetime_last_ride_date, taxi_lifetime_first_ride_date, 
                taxi_lifetime_stage, previous_taxi_lifetime_stage, taxi_income_segment,
                taxi_lifetime_rides, taxi_recency_segment, taxi_rr_active_days_last_21_days as run_taxi_rr_active_days_last_21_days,
                taxi_lifetime_last_ride_city city_name, 
                
                fe_mean_intent_type, fe_intent_trend_type, fe_potential_per_type, 
                fe_regularity_type, fe_recency_type, fe_volatility_type,
                
                gender_tag, ps_tag_link, ps_tag_auto,
                
                taxi_service_affinity, taxi_distance_preference, taxi_time_of_day_preference, taxi_day_of_week_preference
            
            from 
                datasets.iallocator_customer_segments
            where 
                date(run_date) = date('{base_week}')
                -- and taxi_lifetime_last_ride_date = date('2023-11-01')
                and taxi_recency_segment in ('RECENT')
                and taxi_lifetime_rides > 4
                and taxi_lifetime_last_ride_city in ('{city_name}')
        ),
        
        base as (
        
            select 
                a.*,
                case when coalesce(b.rr,0) <1 then 0 else 1 end as rr_retention_flag,
                case when coalesce(b.fe,0) <1 then 0 else 1 end as fe_retention_flag,
                case when coalesce(b.ao,0) <1 then 0 else 1 end as ao_retention_flag,
                case when coalesce(b.nr,0) <1 then 0 else 1 end as net_retention_flag,
                
                case when coalesce(b.nr_post_w01,0) <1 then 0 else 1 end as nr_flag_w01,
                case when coalesce(b.nr_post_w02,0) <1 then 0 else 1 end as nr_flag_w02,
                case when coalesce(b.nr_post_w03,0) <1 then 0 else 1 end as nr_flag_w03,
                case when coalesce(b.nr_post_w04,0) <1 then 0 else 1 end as nr_flag_w04,
                case when coalesce(b.nr_post_w05,0) <1 then 0 else 1 end as nr_flag_w05,
                case when coalesce(b.nr_post_w06,0) <1 then 0 else 1 end as nr_flag_w06,
                case when coalesce(b.nr_post_w07,0) <1 then 0 else 1 end as nr_flag_w07,
                case when coalesce(b.nr_post_w08,0) <1 then 0 else 1 end as nr_flag_w08,
                case when coalesce(b.nr_post_w09,0) <1 then 0 else 1 end as nr_flag_w09,
                case when coalesce(b.nr_post_w10,0) <1 then 0 else 1 end as nr_flag_w10,
                
                case when coalesce(b.rr_post_w01,0) <1 then 0 else 1 end as rr_flag_w01,
                case when coalesce(b.rr_post_w02,0) <1 then 0 else 1 end as rr_flag_w02,
                case when coalesce(b.rr_post_w03,0) <1 then 0 else 1 end as rr_flag_w03,
                case when coalesce(b.rr_post_w04,0) <1 then 0 else 1 end as rr_flag_w04,
                case when coalesce(b.rr_post_w05,0) <1 then 0 else 1 end as rr_flag_w05,
                case when coalesce(b.rr_post_w06,0) <1 then 0 else 1 end as rr_flag_w06,
                case when coalesce(b.rr_post_w07,0) <1 then 0 else 1 end as rr_flag_w07,
                case when coalesce(b.rr_post_w08,0) <1 then 0 else 1 end as rr_flag_w08,
                case when coalesce(b.rr_post_w09,0) <1 then 0 else 1 end as rr_flag_w09,
                case when coalesce(b.rr_post_w10,0) <1 then 0 else 1 end as rr_flag_w10,
                
                case when coalesce(b.fe_post_w01,0) <1 then 0 else 1 end as fe_flag_w01,
                case when coalesce(b.fe_post_w02,0) <1 then 0 else 1 end as fe_flag_w02,
                case when coalesce(b.fe_post_w03,0) <1 then 0 else 1 end as fe_flag_w03,
                case when coalesce(b.fe_post_w04,0) <1 then 0 else 1 end as fe_flag_w04,
                case when coalesce(b.fe_post_w05,0) <1 then 0 else 1 end as fe_flag_w05,
                case when coalesce(b.fe_post_w06,0) <1 then 0 else 1 end as fe_flag_w06,
                case when coalesce(b.fe_post_w07,0) <1 then 0 else 1 end as fe_flag_w07,
                case when coalesce(b.fe_post_w08,0) <1 then 0 else 1 end as fe_flag_w08,
                case when coalesce(b.fe_post_w09,0) <1 then 0 else 1 end as fe_flag_w09,
                case when coalesce(b.fe_post_w10,0) <1 then 0 else 1 end as fe_flag_w10

            from 
                dormant_customer a
            left join
                (
                select 
                    customerid,
                    -- sum(ao_sessions_unique_daily) ao,
                    -- sum(fe_sessions_unique_daily) fe,
                    -- sum(rr_sessions_unique_daily) rr,
                    -- sum(net_rides_daily) nr,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then ao_sessions_unique_daily end) as ao,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr,
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_post_w01,
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_post_w01,
                    
                    
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '70' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w10,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '63' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w09,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '56' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w08,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '49' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w07,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '42' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w06,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '35' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w05,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '28' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w04,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '21' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w03,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '14' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w02,
                    sum(case when day > date_format(date('{base_week}') + interval '0' day, '%Y-%m-%d') and day <= date_format(date('{base_week}') + interval '7' day, '%Y-%m-%d')  then net_rides_daily end) as nr_post_w01

                from 
                    datasets.customer_rf_daily_kpi 
                where 
                    date(day) > date('{base_week}') 
                    and date(day) <= date('{base_week}') + interval '70' day
                    and service_name in ('Link','Auto', 'CabEconomy')
                    and city in ('{city_name}')
                group by 1
                ) b
                on a.customer=b.customerid

        ),
    
        rr_data as (
        
            select 
                fe.*
            from
                (
                select 
                    aa.*, 
                    bb.rr_LT, 
                    bb.rapido_age,
                    bb.rapido_age_seg
                from
                    (
                    select 
                        customerid,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w13,
                        sum(case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w12,
                        sum(case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w11,
                        sum(case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w10,
                        sum(case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w09,
                        sum(case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w08,
                        sum(case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w07,
                        sum(case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w06,
                        sum(case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w05,
                        sum(case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w04,
                        sum(case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w03,
                        sum(case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w02,
                        sum(case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_w01,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then rr_sessions_unique_daily end) as rr_91days,
                        sum(case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then fe_sessions_unique_daily end) as fe_91days,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-92' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d')  then day end) as active_rr_days_w13,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-85' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d')  then day end) as active_rr_days_w12,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-78' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d')  then day end) as active_rr_days_w11,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-71' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d')  then day end) as active_rr_days_w10,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-64' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d')  then day end) as active_rr_days_w09,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-57' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d')  then day end) as active_rr_days_w08,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-50' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d')  then day end) as active_rr_days_w07,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-43' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d')  then day end) as active_rr_days_w06,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-36' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d')  then day end) as active_rr_days_w05,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-29' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d')  then day end) as active_rr_days_w04,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-22' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d')  then day end) as active_rr_days_w03,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-15' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d')  then day end) as active_rr_days_w02,
                        count(distinct case when day > date_format(date('{run_date}') + interval '-08' day, '%Y-%m-%d') and day <= date_format(date('{run_date}') + interval '-01' day, '%Y-%m-%d')  then day end) as active_rr_days_w01,
                        day((date('{run_date}') + interval '-01' day) - date(max(day))) as days_since_last_rr
                    from 
                        (
                        select 
                            customerid,day, 
                            max(fe_sessions_unique_daily) as fe_sessions_unique_daily,  
                            sum(rr_sessions_unique_daily) as rr_sessions_unique_daily
                        from 
                            datasets.customer_rf_daily_kpi a
                        join
                            base b
                            on a.customerid = b.customer
                            and 1 = 1 
                            and city = '{city_name}'
                            and rr_sessions_unique_daily >= 1 
                            and day >= date_format(date('{run_date}') + interval '-91' day , '%Y-%m-%d')
                            and day <= date_format(date('{run_date}') + interval '-1' day , '%Y-%m-%d')
                            and service_name in ('Link','Auto', 'CabEconomy')
                        group by customerid, day
                        )
                    group by customerid
                    ) aa 
                left join 
                    (
                    select 
                        customerid, 
                        sum(rr_sessions_unique_daily) as rr_LT , 
                        day((date('{run_date}') + interval '-01' day) - date(min(day))) as rapido_age,
                        ceil(cast(day((date('{run_date}') + interval '-01' day) - date(min(day))) as real)/10)*10 rapido_age_seg
                    from 
                        datasets.customer_rf_daily_kpi a
                    join
                        base b
                        on a.customerid=b.customer
                        and rr_sessions_unique_daily >= 1 
                        and city = '{city_name}'
                    group by 1
                    ) bb
                    on aa.customerid = bb.customerid
                ) fe 
        )
        
        select 
            a.*, b.*,
            '{run_date}' as run_date
        from 
            base a
        left join
            rr_data b
            on a.customer=b.customerid
    """

    try:
        data = pd.read_sql(rr_data_query, connection3)
    except:
        data = pd.read_sql(rr_data_query, connection4)
        
    
    ## Data fetched successfully
    print(data.shape)


    ## .mask(data[columns_to_replace] == 0, pd.NA) replaces all 0 values in these columns with pd.NA.
    columns_to_replace = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11','active_rr_days_w10', 'active_rr_days_w09',
                          'active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06', 'active_rr_days_w05', 'active_rr_days_w04',
                          'active_rr_days_w03', 'active_rr_days_w02', 'active_rr_days_w01']
    data[columns_to_replace] = data[columns_to_replace].mask(data[columns_to_replace] == 0, pd.NA)
    

    ## Unused
    ## RR-Normalization = Total no. of RR/Active days
    """
    data['rr_norm_w13'] = data.rr_w13/data.active_rr_days_w13
    data['rr_norm_w12'] = data.rr_w12/data.active_rr_days_w12
    data['rr_norm_w11'] = data.rr_w11/data.active_rr_days_w11
    data['rr_norm_w10'] = data.rr_w10/data.active_rr_days_w10
    data['rr_norm_w09'] = data.rr_w09/data.active_rr_days_w09
    data['rr_norm_w08'] = data.rr_w08/data.active_rr_days_w08
    data['rr_norm_w07'] = data.rr_w07/data.active_rr_days_w07
    data['rr_norm_w06'] = data.rr_w06/data.active_rr_days_w06
    data['rr_norm_w05'] = data.rr_w05/data.active_rr_days_w05
    data['rr_norm_w04'] = data.rr_w04/data.active_rr_days_w04
    data['rr_norm_w03'] = data.rr_w03/data.active_rr_days_w03
    data['rr_norm_w02'] = data.rr_w02/data.active_rr_days_w02
    data['rr_norm_w01'] = data.rr_w01/data.active_rr_days_w01
    """
    
    raw_customer_data = data[data['customerid']!=0]
    back_up = raw_customer_data.copy() ## With NA as value
    fe_data2 = raw_customer_data.fillna(0)
    fe_data = fe_data2[fe_data2['customerid']!=0] ## With 0 as value
    
    
    ## Consistency/Regularity Threshold 
    ## RR Regularity → Count of active weeks in last 13 weeks/ (13 or Rapido age in weeks if a customer is less than 13 weeks old)
    ## Active Days Mean → Total active days in last 13 weeks/ count of active weeks in last 13 weeks
    
    ## [OLD version - Unused] 
    """
    def regularity_decider(x,active_mean):
    
        if x<=0.3:
            return '05_rare_need'
        elif x>0.3 and x<=0.5:
            return '04_monthly'
        elif x>0.5 and x<=0.75:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'
        # return 'weekly'
    """

    ## [Relaxed threshold]
    def regularity_decider_v2(x,active_mean):
        
        if x<=0.2:
            return '05_rare_need'
        elif x>0.2 and x<=0.4:
            return '04_monthly'
        elif x>0.4 and x<=0.689:
            return '03_bi_weekly'
        else:
            if active_mean>2.6:
                return '01_daily'
            else:
                return '02_weekly'


    ## Intent Threshold
    ## Intent → Total RR in last 13 weeks/ No. of Active weeks
    def intent_decider(x):
        
        if x<=2:
            return '03_Low'
        elif x>2 and x<=5:
            return '02_Medium'
        else:
            return '01_High'
    

    ## Consideration signal
    def consideration_signal_creator(x,y,active_mean):
        
        if y<=1.5:
            if x<=0.3:
                return 'rarely_1_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_1_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_1_rr'
            else:
                return 'weekly_1_rr'
        else:
            if x<=0.3:
                return 'rarely_2+_rr'
            elif x>0.3 and x<=0.5:
                return 'once_in_4_weeks_2+_rr'
            elif x>0.5 and x<=0.75:
                return 'once_in_2_weeks_2+_rr'
            else:
                return 'weekly_2+_rr'
                
                # Dead-code
                # if active_mean>2.6:
                #     return 'daily'
                # else:
                #     return 'weekly'


    ## Correction factor
    def correction_factor(rr_weight,rapido_age):
        
        if rapido_age >= 91:
            week_inactive = 0
            multiplier = 1

        elif rapido_age == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))

        # 7,14,21,28,...
        elif rapido_age%7 == 0:
            week_inactive = (91-rapido_age)//7 - 1
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2)))    
            
        else:
            week_inactive = (91-rapido_age)//7
            multiplier = (rr_weight.sum()/(rr_weight.sum()- (week_inactive*(week_inactive+1)/2))) 
        
        return (multiplier, 
                week_inactive, 
                (rr_weight.sum()- (week_inactive*(week_inactive+1)/2)), 
                ((rr_weight*rr_weight).sum() - (week_inactive*(week_inactive+1)*(2*week_inactive+1)/6))
               )

    
    ## Recency trend
    def recency_type(days_since_last_fe):
        
        if days_since_last_fe <= 6: 
            return('Recent')
        elif days_since_last_fe <= 20:
            return('Stationary')
        else:
            return('Dormant')

    ## Intent trend
    def intent_trend_type(intent_trend):
        
        if intent_trend <= -0.1:
            return('03_Declining')
        elif intent_trend <= 0.1:
            return('02_Stable')
        else:
            return('01_Increasing')

    # Dead-code
    # def custom_func(mean_intent,rr_trend):
    #     if mean_intent=='High':
    #         return rr_trend
    #     elif mean_intent=='Low' and rr_trend=='01_Increasing':
    #         return '01_Increasing'
    #     else:
    #         return '02_Stable'

    
    
    ## Creating Final signal
    def create_L3_signals(fe_data, back_up): 
        
        print('Creating L3 Signal')
        rr_weight = np.array([1,2,3,4,5,6,7,8,9,10,11,12,13])

        ## sum (rr_sessions_unique_daily)
        fe_cols = ['rr_w13', 'rr_w12', 'rr_w11', 'rr_w10', 'rr_w09','rr_w08', 'rr_w07', 
                   'rr_w06','rr_w05', 'rr_w04', 'rr_w03', 'rr_w02','rr_w01']   
        ## count (rr_days)
        activedays_cols = ['active_rr_days_w13', 'active_rr_days_w12', 'active_rr_days_w11', 'active_rr_days_w10', 
                           'active_rr_days_w09','active_rr_days_w08', 'active_rr_days_w07', 'active_rr_days_w06',
                           'active_rr_days_w05', 'active_rr_days_w04', 'active_rr_days_w03','active_rr_days_w02', 
                           'active_rr_days_w01'] 
        
        ## Unused
        # norm_intent=['rr_norm_w13', 'rr_norm_w12', 'rr_norm_w11', 'rr_norm_w10','rr_norm_w09', 'rr_norm_w08', 'rr_norm_w07', 
        #              'rr_norm_w06','rr_norm_w05', 'rr_norm_w04', 'rr_norm_w03', 'rr_norm_w02','rr_norm_w01']
        # fe_data['fe2rr_91days'] = fe_data.apply(lambda x: (x['rr_91days']/x['fe_91days']) if x['fe_91days'] > 0 else 0.5,axis=1)
        
        
        print('Calctn-1')
        fe_data['rr_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(rr_weight,x)[0])
        ## Unused
        # fe_data['rr_need'] = np.array(fe_data[fe_cols]).max(axis=1)
        # fe_data['no_week_rr_done'] =  back_up[fe_cols].count(axis=1)
        # fe_data['%week_rr_done'] = fe_data.apply(lambda x: x['no_week_rr_done']*(1/(13 - correction_factor(rr_weight,x['rapido_age'])[1])), axis = 1 )
        fe_data['rr_mean_intent'] =  back_up[fe_cols].mean(axis=1) ## Check logic once

        ## Unused
        # fe_data['rr_mean_intent_norm'] =  back_up[norm_intent].mean(axis=1)
        # fe_data['rr_mean_intent_type'] = fe_data['rr_mean_intent'].apply(lambda x: 'Low' if x <= 3 else 'High')
        fe_data['rr_mean_intent_final'] = fe_data['rr_mean_intent'].apply(lambda x: intent_decider(x))

        # Dead-code
        # fe_data['rr_cov'] =  (np.sqrt(back_up[fe_cols].var(axis=1))/back_up[fe_cols].mean(axis=1))
        # fe_data['rr_volatility_type'] = fe_data.apply(lambda x: 'Low' if x['rr_cov'] <= 0.4 else 'High', axis = 1)
        # fe_data['rr_potential'] = fe_data['rr_need'] - fe_data['rr_mean_intent']
        # fe_data['rr_potential_per'] = (fe_data['rr_need'] - fe_data['rr_mean_intent'])/fe_data['rr_mean_intent']
        # fe_data['rr_potential_per_type'] = fe_data['rr_potential_per'].apply(lambda x: 'High' if x >= 0.5 else 'Low')
        
        ## Unused
        # fe_data['rr_trend_n'] = (((13 - fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[1],axis=1)) * (np.array(fe_data[fe_cols])*rr_weight).sum(axis=1))  - (np.array(fe_data[fe_cols]).sum(axis=1) * fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[2],axis=1))) 
        # fe_data['rr_trend_d'] = (((13 - fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[1],axis = 1)) * fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[3],axis =1)) -(fe_data.apply(lambda x: correction_factor(rr_weight,x['rapido_age'])[2],axis =1))**2 )
        # fe_data['rr_trend'] = fe_data['rr_trend_n']/fe_data['rr_trend_d']
        # fe_data['rr_trend'] = fe_data['rr_trend'].fillna(0)
        # fe_data['rr_intent_trend_type'] = fe_data['rr_trend'].apply(lambda x: intent_trend_type(x))
        

        # Notes
        # fe.ge(1.0): This method compares each element in the DataFrame fe to 1.0. 
        # (True if the element is greater than or equal to 1.0, False otherwise).
        # .astype(int): This method converts the boolean values to integers (1 for True and 0 for False).
        fe = fe_data[fe_cols]
        fe = fe.ge(1.0).astype(int)
        
        
        # This multiplies the sum of the weighted values by 1/91 to normalize the result.
        # rr_w13 to rr_w01 || rr_weight 1 to 13
        fe_data['rr_regularity'] = (1/91)*(np.array(fe)*rr_weight).sum(axis=1)
        fe_data['rr_regularity_scaled'] = fe_data['rr_regularity']*fe_data['rr_intent_multiplier']
        
        # Unused
        # fe_data['rr_regularity_type'] = fe_data['rr_regularity_scaled'].apply(lambda x: 'Low' if x <= 0.3 else 'High')
        # fe_data['rr_recency_type'] = fe_data['days_since_last_rr'].apply(lambda x: recency_type(x))
        
        fe_data['active_days_mean'] =  back_up[activedays_cols].mean(axis=1) ## Check logic once
        # fe_data['rr_regularity_final_old'] = fe_data.apply(lambda x: regularity_decider(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        
        print('Calctn-2')
        fe_data['rr_regularity_final'] = fe_data.apply(lambda x: regularity_decider_v2(x['rr_regularity_scaled'],x['active_days_mean']),axis=1)
        fe_data['reg_intent'] = fe_data.rr_regularity_final+'_'+fe_data.rr_mean_intent_final
        
        # print('Calctn-3')
        # fe_data['consideration_final'] = fe_data.apply(lambda x: consideration_signal_creator(x['rr_regularity_scaled'],x['rr_mean_intent'],x['active_days_mean']),axis=1)
        # fe_data['rr_intent_trend_type_v2']= fe_data.apply(lambda x: custom_func(x['rr_mean_intent_type'],x['rr_intent_trend_type']),axis=1)

        return(fe_data)

    
    print('UDF-Created Successfully')
    
    
    L3_signals = create_L3_signals(fe_data,back_up)
    L3_signals['city'] = city_name
    
    final_data = L3_signals.copy()
    
    # final_data = final_data.add_suffix('_nov22')
    # final_data['target']=np.where(final_data.next_lts=='DORMANT',1,0)
    
    return final_data    

## Signal Generation

In [5]:
concatenated_df=pd.DataFrame()

for i in date_list:
    
    print(i)
    tdf = consideration_query(i, city, base_week)
    
    print(tdf.shape[0])
    # concatenated_df=concatenated_df.append(tdf).reset_index(drop=True)
    concatenated_df=pd.concat([concatenated_df, tdf]).reset_index(drop=True)

2024-04-07
(673753, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
649156
2024-04-14
(673753, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
652528
2024-04-21
(673753, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
650503
2024-04-28
(673753, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
650640
2024-05-05
(673753, 93)
UDF-Created Successfully
Creating L3 Signal
Calctn-1
Calctn-2
650808


In [6]:
concatenated_df['reg_intent'] = concatenated_df.rr_regularity_final + '_' + concatenated_df.rr_mean_intent_final
base_df = concatenated_df[concatenated_df.run_date == base_week].reset_index(drop=True)

In [7]:
concatenated_df.head(3)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_intent_multiplier,rr_mean_intent,rr_mean_intent_final,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,632a845b4eaf2597cf0ef57b,333,340.0,2024-04-08,2023-05-10,HOOK,HOOK,UNKNOWN,16,RECENT,2,Bangalore,High,Increasing,High,High,Recent,High,MALE,UNKNOWN,UNKNOWN,LINK_CAB,UNKNOWN,0,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,632a845b4eaf2597cf0ef57b,0.0,1.0,5.0,3.0,6.0,16.0,6.0,4.0,8.0,4.0,4.0,10.0,2.0,69.0,77.0,0.0,1.0,3.0,3.0,3.0,5.0,3.0,4.0,4.0,2.0,3.0,4.0,2.0,5.0,147.0,332.0,340.0,2024-04-07,1.0,5.75,01_High,0.989011,0.989011,3.083333,01_daily,01_daily_01_High,Bangalore
1,6105ab46a318626e41595ad3,980,980.0,2024-04-06,2021-08-01,SUSTENANCE,SOFT_CHURN,HIGH_INCOME,12,RECENT,2,Bangalore,Low,Stable,High,High,Recent,High,MALE,UNKNOWN,UNKNOWN,ONLY_LINK,LONG_DISTANCE,EVENING_PEAK,WEEKEND,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,6105ab46a318626e41595ad3,2.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,3.0,0.0,1.0,8.0,13.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,18.0,99.0,100.0,2024-04-07,1.0,1.60,03_Low,0.384615,0.384615,1.000000,04_monthly,04_monthly_03_Low,Bangalore
2,6583022863d9dccd7bffa7b7,107,110.0,2024-04-04,2023-12-22,COMMITTED,COMMITTED,MEDIUM_INCOME,91,RECENT,13,Bangalore,High,Declining,High,High,Recent,High,MALE,UNKNOWN,PS,ONLY_AUTO,LONG_DISTANCE,MORNING_PEAK,UNKNOWN,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,6583022863d9dccd7bffa7b7,13.0,14.0,13.0,11.0,9.0,0.0,0.0,0.0,0.0,3.0,5.0,9.0,4.0,81.0,113.0,6.0,7.0,6.0,6.0,5.0,0.0,0.0,0.0,0.0,3.0,4.0,5.0,4.0,2.0,251.0,108.0,110.0,2024-04-07,1.0,9.00,01_High,0.670330,0.670330,5.111111,03_bi_weekly,03_bi_weekly_01_High,Bangalore


In [8]:
base_df.head(3)

,customer,rapido_first_ride,rapido_age_seg_fr,taxi_lifetime_last_ride_date,taxi_lifetime_first_ride_date,taxi_lifetime_stage,previous_taxi_lifetime_stage,taxi_income_segment,taxi_lifetime_rides,taxi_recency_segment,run_taxi_rr_active_days_last_21_days,city_name,fe_mean_intent_type,fe_intent_trend_type,fe_potential_per_type,fe_regularity_type,fe_recency_type,fe_volatility_type,gender_tag,ps_tag_link,ps_tag_auto,taxi_service_affinity,taxi_distance_preference,taxi_time_of_day_preference,taxi_day_of_week_preference,rr_retention_flag,fe_retention_flag,ao_retention_flag,net_retention_flag,nr_flag_w01,nr_flag_w02,nr_flag_w03,nr_flag_w04,nr_flag_w05,nr_flag_w06,nr_flag_w07,nr_flag_w08,nr_flag_w09,nr_flag_w10,rr_flag_w01,rr_flag_w02,rr_flag_w03,rr_flag_w04,rr_flag_w05,rr_flag_w06,rr_flag_w07,rr_flag_w08,rr_flag_w09,rr_flag_w10,fe_flag_w01,fe_flag_w02,fe_flag_w03,fe_flag_w04,fe_flag_w05,fe_flag_w06,fe_flag_w07,fe_flag_w08,fe_flag_w09,fe_flag_w10,customerid,rr_w13,rr_w12,rr_w11,rr_w10,rr_w09,rr_w08,rr_w07,rr_w06,rr_w05,rr_w04,rr_w03,rr_w02,rr_w01,rr_91days,fe_91days,active_rr_days_w13,active_rr_days_w12,active_rr_days_w11,active_rr_days_w10,active_rr_days_w09,active_rr_days_w08,active_rr_days_w07,active_rr_days_w06,active_rr_days_w05,active_rr_days_w04,active_rr_days_w03,active_rr_days_w02,active_rr_days_w01,days_since_last_rr,rr_LT,rapido_age,rapido_age_seg,run_date,rr_intent_multiplier,rr_mean_intent,rr_mean_intent_final,rr_regularity,rr_regularity_scaled,active_days_mean,rr_regularity_final,reg_intent,city
0,632a845b4eaf2597cf0ef57b,333,340.0,2024-04-08,2023-05-10,HOOK,HOOK,UNKNOWN,16,RECENT,2,Bangalore,High,Increasing,High,High,Recent,High,MALE,UNKNOWN,UNKNOWN,LINK_CAB,UNKNOWN,0,0,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,0,1,1,1,1,1,1,1,1,1,632a845b4eaf2597cf0ef57b,0.0,1.0,5.0,3.0,6.0,16.0,6.0,4.0,8.0,4.0,4.0,10.0,2.0,69.0,77.0,0.0,1.0,3.0,3.0,3.0,5.0,3.0,4.0,4.0,2.0,3.0,4.0,2.0,5.0,147.0,332.0,340.0,2024-04-07,1.0,5.75,01_High,0.989011,0.989011,3.083333,01_daily,01_daily_01_High,Bangalore
1,6105ab46a318626e41595ad3,980,980.0,2024-04-06,2021-08-01,SUSTENANCE,SOFT_CHURN,HIGH_INCOME,12,RECENT,2,Bangalore,Low,Stable,High,High,Recent,High,MALE,UNKNOWN,UNKNOWN,ONLY_LINK,LONG_DISTANCE,EVENING_PEAK,WEEKEND,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,6105ab46a318626e41595ad3,2.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,3.0,0.0,1.0,8.0,13.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0,1.0,18.0,99.0,100.0,2024-04-07,1.0,1.60,03_Low,0.384615,0.384615,1.000000,04_monthly,04_monthly_03_Low,Bangalore
2,6583022863d9dccd7bffa7b7,107,110.0,2024-04-04,2023-12-22,COMMITTED,COMMITTED,MEDIUM_INCOME,91,RECENT,13,Bangalore,High,Declining,High,High,Recent,High,MALE,UNKNOWN,PS,ONLY_AUTO,LONG_DISTANCE,MORNING_PEAK,UNKNOWN,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,1,6583022863d9dccd7bffa7b7,13.0,14.0,13.0,11.0,9.0,0.0,0.0,0.0,0.0,3.0,5.0,9.0,4.0,81.0,113.0,6.0,7.0,6.0,6.0,5.0,0.0,0.0,0.0,0.0,3.0,4.0,5.0,4.0,2.0,251.0,108.0,110.0,2024-04-07,1.0,9.00,01_High,0.670330,0.670330,5.111111,03_bi_weekly,03_bi_weekly_01_High,Bangalore


In [10]:
concatenated_df.to_csv('concatenated_df.csv', index=False)
base_df.to_csv('base_df.csv', index=False)

# concatenated_df = pd.read_csv('concatenated_df.csv')
# base_df = pd.read_csv('base_df.csv')

## Stability Function

In [11]:
def stability_func(df,base_week):
    
    df_base_week = df[df.run_date == base_week][['customer','run_date','rr_regularity_final','rr_mean_intent_final','reg_intent']]\
                                                    .reset_index(drop=True)
    ranking = {'05_rare_need' : 1,
               '04_monthly' : 2,
               '03_bi_weekly' : 3,
               '02_weekly' : 4,
               '01_daily' : 5
              }

    intent_ranking = {'03_Low' : 1,
                      '02_Medium' : 2,
                      '01_High' : 3
                     }

    final_rank = {'05_rare_need_03_Low' : 1,
                  '05_rare_need_02_Medium' : 2,
                  '05_rare_need_01_High' : 3,
                  '04_monthly_03_Low' : 4,
                  '04_monthly_02_Medium' : 5,
                  '04_monthly_01_High' : 6,
                  '03_bi_weekly_03_Low' : 7,
                  '03_bi_weekly_02_Medium' : 8,
                  '03_bi_weekly_01_High' : 9,
                  '02_weekly_03_Low' : 10,
                  '02_weekly_02_Medium' : 11,
                  '02_weekly_01_High' : 12,
                  '01_daily_02_Medium' : 13,
                  '01_daily_01_High' : 14
                 }    

    df['reg_rank'] = df['rr_regularity_final'].map(ranking)
    df['int_rank'] = df['rr_mean_intent_final'].map(intent_ranking)
    df['final_rank'] = df['reg_intent'].map(final_rank)

    
    
    ## Regularity Stabillity
    pvt_df = pd.pivot_table(df, values='reg_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df.columns
    dropped_reg = pvt_df.iloc[:, 1:].lt(pvt_df[base_week], axis=0)

    # Display the result
    print(dropped_reg.head())
    dropped_reg.head()
    first_dropped_week = dropped_reg.idxmax(axis=1)

    pvt_df['first_dropped_week'] = first_dropped_week.where(dropped_reg.any(axis=1), 'Maintained/Upgraded')

    pvt_df.reset_index(inplace=True)
    pvt_df.head()

    df_base_week_stability = pvt_df.merge(df_base_week, on=['customer'], how='inner')

    reg_stab = df_base_week_stability\
                        .groupby(by=['run_date','rr_regularity_final','first_dropped_week'], as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_regularity_final',], 
                                     columns='first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                   .groupby(by=['rr_regularity_final'],as_index=False)\
                                   .agg(base=('customer','nunique')), on=['rr_regularity_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/regularity_stability.csv',index=False)

    
    
    
    ## Intent Stabillity
    pvt_df_int=pd.pivot_table(df, values='int_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df_int.columns
    dropped_int = pvt_df_int.iloc[:, 1:].lt(pvt_df_int[base_week], axis=0)

    # Display the result
    print(dropped_int.head())
    dropped_int.head()
    int_first_dropped_week = dropped_int.idxmax(axis=1)

    pvt_df_int['int_first_dropped_week'] = int_first_dropped_week.where(dropped_int.any(axis=1), 'Maintained/Upgraded')

    pvt_df_int.reset_index(inplace=True)
    pvt_df_int.head()

    df_base_week_int_stability = pvt_df_int.merge(df_base_week, on=['customer'], how='inner')

    int_stab = df_base_week_int_stability\
                        .groupby(by=['run_date','rr_mean_intent_final','int_first_dropped_week'],as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_mean_intent_final',], 
                                     columns='int_first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                    .groupby(by=['rr_mean_intent_final'], as_index=False)\
                                    .agg(base=('customer','nunique')), on=['rr_mean_intent_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/intent_stability.csv',index=False)

    
    
    
    ## Final Stabillity
    pvt_df_final=pd.pivot_table(df, values='final_rank', index='customer', columns='run_date', aggfunc='sum').reset_index()
    pvt_df_final.columns
    dropped_final = pvt_df_final.iloc[:, 1:].lt(pvt_df_final[base_week], axis=0)

    
    # Display the result
    print(dropped_final.head())
    dropped_final.head()
    final_first_dropped_week = dropped_final.idxmax(axis=1)

    pvt_df_final['final_first_dropped_week'] = final_first_dropped_week.where(dropped_final.any(axis=1), 'Maintained/Upgraded')

    pvt_df_final.reset_index(inplace=True)
    pvt_df_final.head()

    df_base_week_final_stability = pvt_df_final.merge(df_base_week, on=['customer'], how='inner')

    final_stab = df_base_week_final_stability\
                        .groupby(by=['run_date','rr_regularity_final','rr_mean_intent_final','final_first_dropped_week'],
                                 as_index=False)\
                        .agg(user_count=('customer','nunique'))\
                        .pivot_table(values='user_count', 
                                     index=['run_date','rr_regularity_final','rr_mean_intent_final'], 
                                     columns='final_first_dropped_week', 
                                     aggfunc='sum')\
                        .reset_index()\
                        .merge(df_base_week\
                                   .groupby(by=['rr_regularity_final','rr_mean_intent_final'], as_index=False)\
                                   .agg(base=('customer','nunique')), on=['rr_regularity_final','rr_mean_intent_final'], how='inner')\
                        .fillna(0)
    
    # .to_csv(f'{month}/output_files/final_stability.csv',index=False)
    
    return reg_stab, int_stab, final_stab

In [12]:
reg_df, int_df, final_seg_df = stability_func(concatenated_df, base_week)

run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False       False       False       False
2              False       False       False        True       False
3              False       False       False       False       False
4              False       False       False       False        True
run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False       False       False       False
2              False       False       False       False       False
3              False       False       False       False       False
4              False       False       False       False       False
run_date  2024-04-07  2024-04-14  2024-04-21  2024-04-28  2024-05-05
0              False       False       False       False       False
1              False       False  

In [13]:
# complete flow
def stab_func_2(df, base_week, base_df, cons_type):
    
    # base_df=df[df.run_date==base_week].reset_index(drop=True)
    fdf1 = base_df.groupby(by=['city', 'run_date', cons_type], as_index=False).agg(base=('customer','nunique'))
    j = 0
    
    for i in date_list:
        
        j += 1
        if i != base_week:
            
            tdf = df[df.run_date==base_week][['run_date', 'customer', cons_type]]\
                                                .merge(df[df.run_date == i][['customer', cons_type]],on=['customer'],how='left')\
                                                .pivot_table(values='customer', 
                                                             index=['run_date',cons_type+'_x'], 
                                                             columns=cons_type+'_y', aggfunc='nunique')\
                                                .reset_index()\
                                                .fillna(0)
            
            # print(list(tdf))
            items_to_remove = ['run_date', cons_type+'_x']
            filtered_list = [x for x in list(tdf) if x not in items_to_remove]
            tdf.columns = ['run_date', cons_type]+[ j + '_' + i for j in filtered_list]
            fdf1 = fdf1.merge(tdf, on=['run_date', cons_type], how='inner')
    
    return fdf1
        
        

In [14]:
#cons_type: 'rr_regularity_final', 'rr_mean_intent_final', 'reg_intent'
finaldf = stab_func_2(concatenated_df, base_week, base_df, 'reg_intent')

In [15]:
finaldf

,city,run_date,reg_intent,base,01_daily_01_High_2024-04-14,01_daily_02_Medium_2024-04-14,02_weekly_01_High_2024-04-14,02_weekly_02_Medium_2024-04-14,02_weekly_03_Low_2024-04-14,03_bi_weekly_01_High_2024-04-14,03_bi_weekly_02_Medium_2024-04-14,03_bi_weekly_03_Low_2024-04-14,04_monthly_01_High_2024-04-14,04_monthly_02_Medium_2024-04-14,04_monthly_03_Low_2024-04-14,05_rare_need_01_High_2024-04-14,05_rare_need_02_Medium_2024-04-14,05_rare_need_03_Low_2024-04-14,01_daily_01_High_2024-04-21,01_daily_02_Medium_2024-04-21,02_weekly_01_High_2024-04-21,02_weekly_02_Medium_2024-04-21,02_weekly_03_Low_2024-04-21,03_bi_weekly_01_High_2024-04-21,03_bi_weekly_02_Medium_2024-04-21,03_bi_weekly_03_Low_2024-04-21,04_monthly_01_High_2024-04-21,04_monthly_02_Medium_2024-04-21,04_monthly_03_Low_2024-04-21,05_rare_need_01_High_2024-04-21,05_rare_need_02_Medium_2024-04-21,05_rare_need_03_Low_2024-04-21,01_daily_01_High_2024-04-28,01_daily_02_Medium_2024-04-28,02_weekly_01_High_2024-04-28,02_weekly_02_Medium_2024-04-28,02_weekly_03_Low_2024-04-28,03_bi_weekly_01_High_2024-04-28,03_bi_weekly_02_Medium_2024-04-28,03_bi_weekly_03_Low_2024-04-28,04_monthly_01_High_2024-04-28,04_monthly_02_Medium_2024-04-28,04_monthly_03_Low_2024-04-28,05_rare_need_01_High_2024-04-28,05_rare_need_02_Medium_2024-04-28,05_rare_need_03_Low_2024-04-28,01_daily_01_High_2024-05-05,01_daily_02_Medium_2024-05-05,02_weekly_01_High_2024-05-05,02_weekly_02_Medium_2024-05-05,02_weekly_03_Low_2024-05-05,03_bi_weekly_01_High_2024-05-05,03_bi_weekly_02_Medium_2024-05-05,03_bi_weekly_03_Low_2024-05-05,04_monthly_01_High_2024-05-05,04_monthly_02_Medium_2024-05-05,04_monthly_03_Low_2024-05-05,05_rare_need_01_High_2024-05-05,05_rare_need_02_Medium_2024-05-05,05_rare_need_03_Low_2024-05-05
0,Bangalore,2024-04-07,01_daily_01_High,74933,69388.0,2494.0,698.0,610.0,0.0,1687.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,66474.0,3166.0,768.0,1037.0,0.0,2730.0,362.0,0.0,396.0,0.0,0.0,0.0,0.0,0.0,62969.0,3683.0,835.0,1589.0,0.0,4551.0,746.0,0.0,559.0,1.0,0.0,0.0,0.0,0.0,59286.0,4121.0,890.0,2072.0,2.0,6074.0,1265.0,3.0,736.0,141.0,0.0,343.0,0.0,0.0
1,Bangalore,2024-04-07,01_daily_02_Medium,33714,3379.0,25485.0,22.0,3479.0,13.0,28.0,1308.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5465.0,21433.0,37.0,4417.0,10.0,146.0,2007.0,15.0,0.0,184.0,0.0,0.0,0.0,0.0,6480.0,18314.0,53.0,5143.0,28.0,300.0,3108.0,18.0,0.0,270.0,0.0,0.0,0.0,0.0,7142.0,15724.0,57.0,5582.0,32.0,480.0,4057.0,24.0,34.0,407.0,13.0,0.0,162.0,0.0
2,Bangalore,2024-04-07,02_weekly_01_High,4478,690.0,8.0,2436.0,709.0,0.0,604.0,31.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,892.0,12.0,1814.0,852.0,0.0,550.0,143.0,0.0,215.0,0.0,0.0,0.0,0.0,0.0,929.0,17.0,1471.0,909.0,2.0,583.0,260.0,2.0,305.0,0.0,0.0,0.0,0.0,0.0,968.0,27.0,1164.0,905.0,2.0,586.0,353.0,2.0,205.0,66.0,0.0,200.0,0.0,0.0
3,Bangalore,2024-04-07,02_weekly_02_Medium,115429,846.0,3878.0,946.0,93622.0,2246.0,40.0,13573.0,278.0,0.0,0.0,0.0,0.0,0.0,0.0,1822.0,5995.0,1276.0,83247.0,2759.0,198.0,18255.0,988.0,0.0,889.0,0.0,0.0,0.0,0.0,2833.0,6924.0,1409.0,74477.0,3156.0,355.0,22976.0,1676.0,0.0,1612.0,11.0,0.0,0.0,0.0,3804.0,7507.0,1406.0,66105.0,3035.0,578.0,26335.0,2459.0,61.0,3061.0,285.0,0.0,793.0,0.0
4,Bangalore,2024-04-07,02_weekly_03_Low,35781,37.0,40.0,33.0,4548.0,20816.0,0.0,218.0,10089.0,0.0,0.0,0.0,0.0,0.0,0.0,51.0,65.0,15.0,6187.0,15943.0,32.0,1176.0,9665.0,0.0,0.0,2647.0,0.0,0.0,0.0,61.0,84.0,16.0,7147.0,13027.0,31.0,1734.0,10718.0,0.0,4.0,2959.0,0.0,0.0,0.0,65.0,133.0,9.0,7251.0,10111.0,13.0,2589.0,10376.0,29.0,321.0,2516.0,0.0,0.0,2368.0
5,Bangalore,2024-04-07,03_bi_weekly_01_High,9165,1942.0,150.0,260.0,158.0,0.0,5510.0,799.0,1.0,322.0,23.0,0.0,0.0,0.0,0.0,3077.0,219.0,298.0,329.0,0.0,3687.0,975.0,4.0,510.0,63.0,3.0,0.0,0.0,0.0,3322.0,297.0,330.0,412.0,0.0,2736.0,1085.0,13.0,796.0,159.0,14.0,1.0,0.0,0.0,3344.0,366.0,338.0,552.0,3.0,2169.0,1057.0,27.0,941.0,289.0,21.0,36.0,17.0,5.0
6,Bangalore,2024-04-07,03_bi_weekly_02_Medium,91498,280.0,1508.0,119.0,13701.0,657.0,1038.0,64252.0,3643.0,31.0,5970.0,299.0,0.0,0.0,

In [16]:
base_df\
.groupby(by=['city_name','run_date','reg_intent'],as_index=False)\
.agg(
    user_count=('customer','nunique'),
    fe_retention_w01=('fe_flag_w01','sum'),
    fe_retention_w02=('fe_flag_w02','sum'),
    fe_retention_w03=('fe_flag_w03','sum'),
    fe_retention_w04=('fe_flag_w04','sum'),
    fe_retention_w05=('fe_flag_w05','sum'),
    fe_retention_w06=('fe_flag_w06','sum'),
    fe_retention_w07=('fe_flag_w07','sum'),
    fe_retention_w08=('fe_flag_w08','sum'),
    fe_retention_w09=('fe_flag_w09','sum'),
    fe_retention_w10=('fe_flag_w10','sum'),
    
    rr_retention_w01=('rr_flag_w01','sum'),
    rr_retention_w02=('rr_flag_w02','sum'),
    rr_retention_w03=('rr_flag_w03','sum'),
    rr_retention_w04=('rr_flag_w04','sum'),
    rr_retention_w05=('rr_flag_w05','sum'),
    rr_retention_w06=('rr_flag_w06','sum'),
    rr_retention_w07=('rr_flag_w07','sum'),
    rr_retention_w08=('rr_flag_w08','sum'),
    rr_retention_w09=('rr_flag_w09','sum'),
    rr_retention_w10=('rr_flag_w10','sum'),
        
    nr_retention_w01=('nr_flag_w01','sum'),
    nr_retention_w02=('nr_flag_w02','sum'),
    nr_retention_w03=('nr_flag_w03','sum'),
    nr_retention_w04=('nr_flag_w04','sum'),
    nr_retention_w05=('nr_flag_w05','sum'),
    nr_retention_w06=('nr_flag_w06','sum'),
    nr_retention_w07=('nr_flag_w07','sum'),
    nr_retention_w08=('nr_flag_w08','sum'),
    nr_retention_w09=('nr_flag_w09','sum'),
    nr_retention_w10=('nr_flag_w10','sum')
    )


# .to_csv(f'{month}/output_files/reg_retention_10w_newlogic.csv',index=False)

#rr_mean_intent_final, rr_regularity_final, reg_intent

# gender_tag,
# taxi_income_segment,
# ps_tag_link,
# ps_tag_auto,
# taxi_service_affinity,
# taxi_distance_preference,
# taxi_time_of_day_preference,
# taxi_day_of_week_preference

,city_name,run_date,reg_intent,user_count,fe_retention_w01,fe_retention_w02,fe_retention_w03,fe_retention_w04,fe_retention_w05,fe_retention_w06,fe_retention_w07,fe_retention_w08,fe_retention_w09,fe_retention_w10,rr_retention_w01,rr_retention_w02,rr_retention_w03,rr_retention_w04,rr_retention_w05,rr_retention_w06,rr_retention_w07,rr_retention_w08,rr_retention_w09,rr_retention_w10,nr_retention_w01,nr_retention_w02,nr_retention_w03,nr_retention_w04,nr_retention_w05,nr_retention_w06,nr_retention_w07,nr_retention_w08,nr_retention_w09,nr_retention_w10
0,Bangalore,2024-04-07,01_daily_01_High,74933,67009,71303,72325,72878,73198,73401,73542,73641,73732,73785,65903,70682,71840,72502,72862,73103,73278,73402,73522,73587,61622,68545,70467,71494,72086,72469,72725,72896,73051,73144
1,Bangalore,2024-04-07,01_daily_02_Medium,33714,28829,31788,32426,32716,32894,33008,33087,33145,33191,33213,28014,31332,32083,32447,32667,32820,32925,32989,33049,33084,24473,29377,30835,31530,31963,32216,32408,32532,32656,32736
2,Bangalore,2024-04-07,02_weekly_01_High,4478,3294,3806,3935,4021,4064,4088,4107,4125,4149,4161,3132,3691,3837,3928,3983,4019,4047,4069,4091,4108,2802,3464,3677,3799,3875,3926,3959,3989,4022,4044
3,Bangalore,2024-04-07,02_weekly_02_Medium,115429,87831,103066,107421,109536,110704,111377,111809,112161,112419,112610,82699,99900,105160,107872,109415,110316,110889,111342,111668,111919,69346,90297,98343,102812,105482,107097,108241,109059,109722,110229
4,Bangalore,2024-04-07,02_weekly_03_Low,35781,22189,28094,30125,31203,31839,32217,32453,32656,32826,32942,19853,26168,28602,29992,30840,31323,31653,31937,32151,32319,15732,22410,25516,27481,28761,29591,30171,30606,30971,31262
5,Bangalore,2024-04-07,03_bi_weekly_01_High,9165,7031,7968,8252,8445,8578,8659,8724,8765,8807,8841,6739,7756,8085,8292,8447,8540,8622,8674,8724,8767,6164,7333,7752,8025,8221,8342,8454,8513,8577,8633
6,Bangalore,2024-04-07,03_bi_weekly_02_Medium,91498,60172,75073,80492,83579,85507,86624,87361,87940,88349,88682,54741,70626,76919,80730,83177,84661,85677,86531,87073,87559,44474,61202,69152,74353,77821,80156,81830,83118,84141,85005
7,Bangalore,2024-04-07,03_bi_weekly_03_Low,92997,51212,69574,77415,81976,84829,86611,87744,88639,89272,89716,44084,62568,71382,77018,80740,83089,84707,86042,86978,87732,33490,50960,60799,67625,72442,75752,78378,80418,81947,83249
8,Bangalore,2024-04-07,04_monthly_01_High,4116,2688,3080,3240,3388,3502,3562,3616,3680,3702,3725,2540,2940,3119,3270,3396,3467,3534,3593,3620,3655,2331,2746,2947,3125,3254,3332,3413,3488,3533,3580
9,Bangalore,2024-04-07,04_monthly_02_Medium,38675,20400,26591,29581,31540,32905,33807,34411,34950,35312,35598,17732,23953,27150,29351,30987,32099,32871,33568,34076,34498,14215,20188,23564,26099,27942,29327,30434,31302,31998,32619


In [17]:
test=list(finaldf)

In [18]:
# test.remove([['city','01_daily_2023-11-11']])

In [19]:
test=['city']+[i+'_yo' for i in test]

In [20]:
test

['city',
 'city_yo',
 'run_date_yo',
 'reg_intent_yo',
 'base_yo',
 '01_daily_01_High_2024-04-14_yo',
 '01_daily_02_Medium_2024-04-14_yo',
 '02_weekly_01_High_2024-04-14_yo',
 '02_weekly_02_Medium_2024-04-14_yo',
 '02_weekly_03_Low_2024-04-14_yo',
 '03_bi_weekly_01_High_2024-04-14_yo',
 '03_bi_weekly_02_Medium_2024-04-14_yo',
 '03_bi_weekly_03_Low_2024-04-14_yo',
 '04_monthly_01_High_2024-04-14_yo',
 '04_monthly_02_Medium_2024-04-14_yo',
 '04_monthly_03_Low_2024-04-14_yo',
 '05_rare_need_01_High_2024-04-14_yo',
 '05_rare_need_02_Medium_2024-04-14_yo',
 '05_rare_need_03_Low_2024-04-14_yo',
 '01_daily_01_High_2024-04-21_yo',
 '01_daily_02_Medium_2024-04-21_yo',
 '02_weekly_01_High_2024-04-21_yo',
 '02_weekly_02_Medium_2024-04-21_yo',
 '02_weekly_03_Low_2024-04-21_yo',
 '03_bi_weekly_01_High_2024-04-21_yo',
 '03_bi_weekly_02_Medium_2024-04-21_yo',
 '03_bi_weekly_03_Low_2024-04-21_yo',
 '04_monthly_01_High_2024-04-21_yo',
 '04_monthly_02_Medium_2024-04-21_yo',
 '04_monthly_03_Low_2024-04-2

In [22]:
finaldf.to_csv('stability_v0.csv', index=False)

In [23]:
finaldf.head(2)

,city,run_date,reg_intent,base,01_daily_01_High_2024-04-14,01_daily_02_Medium_2024-04-14,02_weekly_01_High_2024-04-14,02_weekly_02_Medium_2024-04-14,02_weekly_03_Low_2024-04-14,03_bi_weekly_01_High_2024-04-14,03_bi_weekly_02_Medium_2024-04-14,03_bi_weekly_03_Low_2024-04-14,04_monthly_01_High_2024-04-14,04_monthly_02_Medium_2024-04-14,04_monthly_03_Low_2024-04-14,05_rare_need_01_High_2024-04-14,05_rare_need_02_Medium_2024-04-14,05_rare_need_03_Low_2024-04-14,01_daily_01_High_2024-04-21,01_daily_02_Medium_2024-04-21,02_weekly_01_High_2024-04-21,02_weekly_02_Medium_2024-04-21,02_weekly_03_Low_2024-04-21,03_bi_weekly_01_High_2024-04-21,03_bi_weekly_02_Medium_2024-04-21,03_bi_weekly_03_Low_2024-04-21,04_monthly_01_High_2024-04-21,04_monthly_02_Medium_2024-04-21,04_monthly_03_Low_2024-04-21,05_rare_need_01_High_2024-04-21,05_rare_need_02_Medium_2024-04-21,05_rare_need_03_Low_2024-04-21,01_daily_01_High_2024-04-28,01_daily_02_Medium_2024-04-28,02_weekly_01_High_2024-04-28,02_weekly_02_Medium_2024-04-28,02_weekly_03_Low_2024-04-28,03_bi_weekly_01_High_2024-04-28,03_bi_weekly_02_Medium_2024-04-28,03_bi_weekly_03_Low_2024-04-28,04_monthly_01_High_2024-04-28,04_monthly_02_Medium_2024-04-28,04_monthly_03_Low_2024-04-28,05_rare_need_01_High_2024-04-28,05_rare_need_02_Medium_2024-04-28,05_rare_need_03_Low_2024-04-28,01_daily_01_High_2024-05-05,01_daily_02_Medium_2024-05-05,02_weekly_01_High_2024-05-05,02_weekly_02_Medium_2024-05-05,02_weekly_03_Low_2024-05-05,03_bi_weekly_01_High_2024-05-05,03_bi_weekly_02_Medium_2024-05-05,03_bi_weekly_03_Low_2024-05-05,04_monthly_01_High_2024-05-05,04_monthly_02_Medium_2024-05-05,04_monthly_03_Low_2024-05-05,05_rare_need_01_High_2024-05-05,05_rare_need_02_Medium_2024-05-05,05_rare_need_03_Low_2024-05-05
0,Bangalore,2024-04-07,01_daily_01_High,74933,69388.0,2494.0,698.0,610.0,0.0,1687.0,56.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,66474.0,3166.0,768.0,1037.0,0.0,2730.0,362.0,0.0,396.0,0.0,0.0,0.0,0.0,0.0,62969.0,3683.0,835.0,1589.0,0.0,4551.0,746.0,0.0,559.0,1.0,0.0,0.0,0.0,0.0,59286.0,4121.0,890.0,2072.0,2.0,6074.0,1265.0,3.0,736.0,141.0,0.0,343.0,0.0,0.0
1,Bangalore,2024-04-07,01_daily_02_Medium,33714,3379.0,25485.0,22.0,3479.0,13.0,28.0,1308.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,5465.0,21433.0,37.0,4417.0,10.0,146.0,2007.0,15.0,0.0,184.0,0.0,0.0,0.0,0.0,6480.0,18314.0,53.0,5143.0,28.0,300.0,3108.0,18.0,0.0,270.0,0.0,0.0,0.0,0.0,7142.0,15724.0,57.0,5582.0,32.0,480.0,4057.0,24.0,34.0,407.0,13.0,0.0,162.0,0.0


## Notes

### Explanation of correction_factor Function

The correction_factor function takes two parameters: rr_weight and rapido_age.

Parameters:

rr_weight: A series or array of weights (assumed to be a pandas Series based on the usage).
rapido_age: An integer representing the age in days.

Logic and Calculations:
The function performs different calculations based on the value of rapido_age.

    Condition 1: rapido_age >= 91:
        week_inactive is set to 0.
        multiplier is set to 1.
        This means there is no inactivity, and the multiplier is 1 (no correction).
    
    Condition 2: rapido_age == 0:
        week_inactive is calculated as (91 - rapido_age) // 7 - 1.
        multiplier is calculated using the formula:
            multiplier = rr_weight.sum() / (rr_weight.sum() - (week_inactive * (week_inactive + 1) / 2))
        This corrects for the inactivity based on the number of weeks inactive.
    
    Condition 3: rapido_age % 7 == 0:
        week_inactive is calculated similarly to Condition 2.
        multiplier is calculated using the same formula as in Condition 2.
    
    Condition 4: All Other Cases:
        week_inactive is calculated as (91 - rapido_age) // 7.
        multiplier is calculated using the same formula as in Conditions 2 and 3.
        
    Return Values:
        The function returns a tuple containing:
            multiplier: The correction factor.
            week_inactive: Number of inactive weeks.
            Corrected sum of weights: rr_weight.sum() - (week_inactive * (week_inactive + 1) / 2).
            Corrected sum of squared weights: (rr_weight * rr_weight).sum() - (week_inactive * (week_inactive + 1) * (2 * week_inactive + 1) / 6).

                
Applying correction_factor to fe_data
    The line fe_data['rr_intent_multiplier'] = fe_data['rapido_age'].apply(lambda x: correction_factor(rr_weight, x)[0]) adds a new column rr_intent_multiplier to the DataFrame fe_data.

- For each value in the column rapido_age of fe_data, the correction_factor function is called.
- The first element of the returned tuple (i.e., multiplier) is extracted and assigned to the new column rr_intent_multiplier.

In [24]:
rapido_age = 0
rr_weight = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])

rr_weight_sum = rr_weight.sum()
week_inactive = (91 - rapido_age) // 7 - 1
correction = (week_inactive * (week_inactive + 1)) // 2
adjusted_sum = rr_weight_sum - correction
multiplier = rr_weight_sum / adjusted_sum

print("rr_weight_sum:", rr_weight_sum)
print("week_inactive:", week_inactive)
print("correction:", correction)
print("adjusted_sum:", adjusted_sum)
print("multiplier:", multiplier)

rr_weight_sum: 91
week_inactive: 12
correction: 78
adjusted_sum: 13
multiplier: 7.0


Explanation of Logic

week_inactive: Number of weeks of inactivity calculated based on rapido_age.

correction: Sum of an arithmetic series representing the cumulative impact of inactive weeks.

adjusted_sum: The total sum of rr_weight after adjusting for inactivity.

multiplier: The correction factor used to adjust the weights considering inactivity.

This correction factor ensures that the impact of inactive weeks is accounted for by adjusting the total weight proportionally.

In [25]:
import numpy as np

rr_weight = np.array([1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13])
rr_weight_sum = rr_weight.sum()

rapido_ages = list(range(0, 91, 5))  # List of rapido_age values from 0 to 90 in increments of 5

for rapido_age in rapido_ages:
    if rapido_age >= 91:
        week_inactive = 0
        correction = 0
    else:
        week_inactive = (91 - rapido_age) // 7 - 1
        correction = (week_inactive * (week_inactive + 1)) // 2

    adjusted_sum = rr_weight_sum - correction
    multiplier = rr_weight_sum / adjusted_sum

    print(f"rapido_age: {rapido_age}")
    print("week_inactive:", week_inactive)
    print("correction:", correction)
    print("adjusted_sum:", adjusted_sum)
    print("multiplier:", multiplier)
    print("-" * 40)

rapido_age: 0
week_inactive: 12
correction: 78
adjusted_sum: 13
multiplier: 7.0
----------------------------------------
rapido_age: 5
week_inactive: 11
correction: 66
adjusted_sum: 25
multiplier: 3.64
----------------------------------------
rapido_age: 10
week_inactive: 10
correction: 55
adjusted_sum: 36
multiplier: 2.5277777777777777
----------------------------------------
rapido_age: 15
week_inactive: 9
correction: 45
adjusted_sum: 46
multiplier: 1.9782608695652173
----------------------------------------
rapido_age: 20
week_inactive: 9
correction: 45
adjusted_sum: 46
multiplier: 1.9782608695652173
----------------------------------------
rapido_age: 25
week_inactive: 8
correction: 36
adjusted_sum: 55
multiplier: 1.6545454545454545
----------------------------------------
rapido_age: 30
week_inactive: 7
correction: 28
adjusted_sum: 63
multiplier: 1.4444444444444444
----------------------------------------
rapido_age: 35
week_inactive: 7
correction: 28
adjusted_sum: 63
multiplier: 

Interpretation

rapido_age = 0: The maximum number of inactive weeks (12 weeks) results in a correction of 78, leading to a high multiplier of 7.0.

rapido_age = 5: The number of inactive weeks decreases to 11, with a correction of 66, resulting in a multiplier of approximately 3.64.

rapido_age = 10 to 90: As rapido_age increases, the number of inactive weeks and the corresponding correction decrease, leading to lower multipliers.

For rapido_age values from 80 to 90, the multiplier becomes 1.0, indicating no correction is needed since there are no inactive weeks.